V tomto sešitu najdeme nejfrekventovanější úseky napříč republikou (vždy jen 1 pro 1 město) a vyexportujeme je do souborů ```.yaml``` ve složce ```data```.

In [12]:
import os
import yaml
import polars as pl
import altair as alt

In [13]:
pl.Config(tbl_rows=100)
pl.Config(fmt_str_lengths=150)
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [14]:
df = pl.read_parquet(
    "data/segments.parquet"
).sort(
    ["id", "date"]
).with_columns(
    days_diff = pl.col("date").diff().dt.total_days().over("id"),
    effort_diff = pl.col("effort_count").diff().over("id")
).with_columns(
    avg_daily_efforts = pl.col("effort_diff") / pl.col("days_diff")
).with_columns(
    pl.when(pl.col("city").str.contains("Praha") | pl.col("city").str.contains("Prague")).then(pl.lit('Praha')).when(pl.col('city').str.contains('Brno')).then(pl.lit('Brno')).otherwise(pl.col('city')).alias('city'),
    pl.when(pl.col("effort_diff") < 0).then(pl.lit(0)).otherwise(pl.col("effort_diff")).alias("effort_diff")
).with_columns(
    pl.col('date').dt.ordinal_day().alias('day'),
    pl.col('date').dt.week().alias('week'),
    pl.col('date').dt.year().alias('year')
)

## Rozsah dat

In [16]:
print(df.select(pl.col("date").min()).item())
print(df.select(pl.col("date").max()).item())

2024-12-24 23:05:01+01:00
2026-03-09 23:38:56+01:00


In [17]:
df.sample(1)

id,name,activity_type,distance,average_grade,elevation_high,elevation_low,total_elevation_gain,start_latlng,end_latlng,country,state,city,effort_count,athlete_count,date,start_to_finish_distance,days_diff,effort_diff,avg_daily_efforts,day,week,year
i64,str,str,f64,f64,f64,f64,f64,list[f64],list[f64],str,str,str,i64,i64,"datetime[μs, CET]",f64,i64,i64,f64,i16,i8,i32
8646352,"""Kopeček - od kruháče k bazilice""","""Run""",776.1,14.1,366.4,256.6,109.8,"[49.625989, 17.327876]","[49.629042, 17.337408]","""Czechia""","""Olomouc Region""","""Samotišky""",4057,498,2026-01-25 23:38:48 CET,0.765906,1,1,1.0,25,4,2026


## Výběr segmentů ke sledování

### Nezávazný žebříček

In [20]:
zebricek = df.filter(
    pl.col("country") == "Czechia"
).group_by(
    ["name","city","activity_type","start_to_finish_distance"]
).agg(
    pl.col("effort_diff").quantile(0.8)
).sort(
    by="effort_diff", descending=True
)

zebricek.head(100)

name,city,activity_type,start_to_finish_distance,effort_diff
str,str,str,f64,f64
"""Die Eine Runde""","""Brno""","""Ride""",0.002446,419.0
"""VUT horní dráha 400m""","""Brno""","""Run""",0.011266,219.0
"""Opičí Časovka, Podolská vodárna - Dvorce""","""Praha""","""Ride""",1.379737,210.0
"""Stadion Na Děkance""","""Praha""","""Run""",0.006087,187.0
"""Zweite hup u ZOO""","""Praha""","""Ride""",0.308372,184.0
"""Zbraslav - Komořany""","""Zbraslav""","""Ride""",2.258969,151.0
"""Pražačka 320""","""Praha""","""Run""",0.000925,140.0
"""Šlechtovka - finální rovinka""","""Praha""","""Run""",0.222849,139.0
"""Lužánky rovinka""","""Brno""","""Run""",0.260571,123.0


### Finální seznam: běh

In [22]:
celorepublika = zebricek.filter(
    (pl.col('activity_type') == 'Run') & (pl.col('start_to_finish_distance') > 0.5)
).unique(
    subset='city',keep='first'
).sort(
    by='effort_diff',descending=True
).head(
    20
)

print(celorepublika)

celorepublika = celorepublika.select(pl.col('name')).to_series().to_list()

with open("data/behani_top_20.yaml", "w", encoding="utf-8") as f:
    yaml.dump(celorepublika, f, allow_unicode=True, default_flow_style=False)

shape: (20, 5)
┌───────────────────────────────────┬─────────────────────────┬───────────────┬──────────────────────────┬─────────────┐
│ name                              ┆ city                    ┆ activity_type ┆ start_to_finish_distance ┆ effort_diff │
│ ---                               ┆ ---                     ┆ ---           ┆ ---                      ┆ ---         │
│ str                               ┆ str                     ┆ str           ┆ f64                      ┆ f64         │
╞═══════════════════════════════════╪═════════════════════════╪═══════════════╪══════════════════════════╪═════════════╡
│ tenis 1K                          ┆ Brno                    ┆ Run           ┆ 0.951894                 ┆ 45.0        │
│ Smetanovy reverse                 ┆ Olomouc                 ┆ Run           ┆ 0.631364                 ┆ 41.0        │
│ Hvězda flat                       ┆ Praha                   ┆ Run           ┆ 0.995211                 ┆ 33.0        │
│ Stezka nábřeží 

### Finální seznam: kolo

In [24]:
celorepublika_kolo = zebricek.filter(
    (pl.col('activity_type') == 'Ride') & (pl.col('start_to_finish_distance') > 1)
).unique(
    subset='city',keep='first'
).sort(
    by='effort_diff',descending=True
).head(
    20
)

print(celorepublika_kolo)

celorepublika_kolo = celorepublika_kolo.select(pl.col('name')).to_series().to_list()

with open("data/kolo_top_20.yaml", "w", encoding="utf-8") as f:
    yaml.dump(celorepublika_kolo, f, allow_unicode=True, default_flow_style=False)

shape: (20, 5)
┌──────────────────────────────────┬──────────────────────────┬───────────────┬──────────────────────────┬─────────────┐
│ name                             ┆ city                     ┆ activity_type ┆ start_to_finish_distance ┆ effort_diff │
│ ---                              ┆ ---                      ┆ ---           ┆ ---                      ┆ ---         │
│ str                              ┆ str                      ┆ str           ┆ f64                      ┆ f64         │
╞══════════════════════════════════╪══════════════════════════╪═══════════════╪══════════════════════════╪═════════════╡
│ Opičí Časovka, Podolská vodárna  ┆ Praha                    ┆ Ride          ┆ 1.379737                 ┆ 210.0       │
│ - Dvorce                         ┆                          ┆               ┆                          ┆             │
│ Zbraslav - Komořany              ┆ Zbraslav                 ┆ Ride          ┆ 2.258969                 ┆ 151.0       │
│ time trial at S

In [25]:
celorepublika_kolo_30 = zebricek.filter(
    (pl.col('activity_type') == 'Ride') & (pl.col('start_to_finish_distance') > 1)
).unique(
    subset='city',keep='first'
).sort(
    by='effort_diff',descending=True
).head(
    30
)

print(celorepublika_kolo_30)

celorepublika_kolo_30 = celorepublika_kolo_30.select(pl.col('name')).to_series().to_list()

with open("data/kolo_top_30.yaml", "w", encoding="utf-8") as f:
    yaml.dump(celorepublika_kolo_30, f, allow_unicode=True, default_flow_style=False)

shape: (30, 5)
┌──────────────────────────────┬──────────────────────────────┬───────────────┬──────────────────────────┬─────────────┐
│ name                         ┆ city                         ┆ activity_type ┆ start_to_finish_distance ┆ effort_diff │
│ ---                          ┆ ---                          ┆ ---           ┆ ---                      ┆ ---         │
│ str                          ┆ str                          ┆ str           ┆ f64                      ┆ f64         │
╞══════════════════════════════╪══════════════════════════════╪═══════════════╪══════════════════════════╪═════════════╡
│ Opičí Časovka, Podolská      ┆ Praha                        ┆ Ride          ┆ 1.379737                 ┆ 210.0       │
│ vodárna - Dvorce             ┆                              ┆               ┆                          ┆             │
│ Zbraslav - Komořany          ┆ Zbraslav                     ┆ Ride          ┆ 2.258969                 ┆ 151.0       │
│ time trial at S

### Finální seznam: běžecké ovály

In [27]:
ovaly = df.filter(
    (pl.col('country') == 'Czechia') & (pl.col('activity_type') == 'Run') & (pl.col('start_to_finish_distance') <= 0.02) & (pl.col('distance') < 450)
).group_by(
    ["name","start_to_finish_distance","distance"]
).agg(
    pl.col('effort_diff').mean()
).sort(
    by='effort_diff',descending=True
).filter(
    pl.col('effort_diff') > 7
)

print(ovaly)

ovaly = ovaly.select(pl.col('name')).to_series().to_list()

with open("data/ovaly.yaml", "w", encoding="utf-8") as f:
    yaml.dump(ovaly, f, allow_unicode=True, default_flow_style=False)

shape: (20, 4)
┌──────────────────────────────────────┬──────────────────────────┬──────────┬─────────────┐
│ name                                 ┆ start_to_finish_distance ┆ distance ┆ effort_diff │
│ ---                                  ┆ ---                      ┆ ---      ┆ ---         │
│ str                                  ┆ f64                      ┆ f64      ┆ f64         │
╞══════════════════════════════════════╪══════════════════════════╪══════════╪═════════════╡
│ Stadion Na Děkance                   ┆ 0.006087                 ┆ 420.9    ┆ 102.817308  │
│ VUT horní dráha 400m                 ┆ 0.011266                 ┆ 405.2    ┆ 98.302158   │
│ Pražačka 320                         ┆ 0.000925                 ┆ 343.3    ┆ 92.048077   │
│ TJ Sokol okruh                       ┆ 0.009342                 ┆ 418.8    ┆ 46.183771   │
│ 400m-Liberec                         ┆ 0.014369                 ┆ 408.0    ┆ 42.264851   │
│ Strahovská dráha                     ┆ 0.001716      

### Finální seznam: cyklosegmenty nad 1000 m. n. m.

In [29]:
nad1000 = df.filter(
    (pl.col("activity_type") == "Ride") & (pl.col('country') == 'Czechia')
).group_by(
    "name"
).agg(
    pl.col("elevation_high").max(),
    pl.col("effort_diff").mean()
).sort(
    by="elevation_high",descending=True
).filter(
    pl.col('effort_diff') > 2
).filter(
    pl.col('elevation_high') >= 1000
)

print(nad1000)

nad1000 = nad1000.select(pl.col('name')).to_series().to_list()

with open("data/nad_1000_mnm.yaml", "w", encoding="utf-8") as f:
    yaml.dump(nad1000, f, allow_unicode=True, default_flow_style=False)

shape: (10, 3)
┌────────────────────────────────────────────────────┬────────────────┬─────────────┐
│ name                                               ┆ elevation_high ┆ effort_diff │
│ ---                                                ┆ ---            ┆ ---         │
│ str                                                ┆ f64            ┆ f64         │
╞════════════════════════════════════════════════════╪════════════════╪═════════════╡
│ Sedýlko - Praděd climb                             ┆ 1487.0         ┆ 10.327014   │
│ Zlaťák konec - smrt                                ┆ 1383.2         ┆ 4.07381     │
│ Lysá hora - 5 km                                   ┆ 1308.5         ┆ 9.175772    │
│ Výstup na Klínovec                                 ┆ 1241.8         ┆ 6.59375     │
│ Klinovec Top                                       ┆ 1238.8         ┆ 8.57619     │
│ Špindlerovka - část II. (od Černýho potoka nahoru) ┆ 1194.4         ┆ 3.593301    │
│ Do sedla Klínovce po silnici         